# Sequence-Space Jacobians for MarkovConsumerType

This notebook demonstrates computing impulse response functions (IRFs) via
the Fake News Algorithm (Auclert et al. 2021) applied to the Markov
consumption-saving model.

We:
1. Compute the partial-equilibrium steady state using TM methods
2. Compute Jacobians of aggregate C and A w.r.t. an Rfree shock
3. Verify against finite-difference numerical derivatives from TM propagation
4. Compare the Markov J=1 Jacobian shape with the NK model's Jacobian


In [1]:
import time
import numpy as np
from copy import deepcopy
from HARK.ConsumptionSaving.ConsMarkovModel import (
    MarkovConsumerType,
    init_indshk_markov,
)

COLOR_MC = "tab:blue"
COLOR_TM = "tab:orange"

## 1. Set up and solve the steady state

Calibration follows `init_indshk_markov` from `ConsMarkovModel`, which
extends the baseline `IndShockConsumerType` calibration with a symmetric
two-state Markov chain (p11 = p22 = 0.9). Interest rates, survival
probabilities, and permanent income growth are set equal across states so
the Markov structure is active but states are symmetric — isolating the
effect of the transition-matrix machinery from state-dependent economics.

In [2]:
params = deepcopy(init_indshk_markov)
params["Mrkv_p11"] = [0.9]
params["Mrkv_p22"] = [0.9]
# Symmetric across Markov states — no state-dependent economics
params["Rfree"] = [np.array([1.03, 1.03])]
params["LivPrb"] = [np.array([0.98, 0.98])]
params["PermGroFac"] = [np.array([1.0, 1.0])]
params["cycles"] = 0  # infinite-horizon

agent = MarkovConsumerType(**params)
# Solves the model, builds the TM, and finds the ergodic distribution
A_ss, C_ss = agent.compute_pe_steady_state()
print(f"Steady state: A_ss = {A_ss:.6f}, C_ss = {C_ss:.6f}")

# Column sums of 1.0 confirm the TM is a valid probability matrix
col_sums = agent.tran_matrix.sum(axis=0)
print(f"TM column sums: min={col_sums.min():.12f}, max={col_sums.max():.12f}")

Steady state: A_ss = 0.835777, C_ss = 1.007856
TM column sums: min=1.000000000000, max=1.000000000000


## 2. Compute Jacobians via Fake News Algorithm

In [3]:
T = 50  # Jacobian horizon: 50 periods

t0 = time.time()
# Fake News Algorithm (Auclert et al. 2021): decomposes the Jacobian into
# curly-D, curly-P, and fake-news matrix components for efficiency.
CJAC, AJAC = agent.calc_jacobian("Rfree", T)
jac_time = time.time() - t0
print(f"Jacobians computed in {jac_time:.1f} seconds")

# Column s of AJAC gives the IRF of aggregate A to a one-period
# Rfree shock at date s. Column 0 = MIT shock at t=0.
print(f"AJAC shape: {AJAC.shape}")
print("IRF of A to Rfree shock (first 10 periods):")
for t in range(min(10, T)):
    print(f"  t={t:2d}: dA/dR = {AJAC[t, 0]:>10.4f}")

Jacobians computed in 0.3 seconds
AJAC shape: (50, 50)
IRF of A to Rfree shock (first 10 periods):
  t= 0: dA/dR =     1.0131
  t= 1: dA/dR =     0.9033
  t= 2: dA/dR =     0.8077
  t= 3: dA/dR =     0.7239
  t= 4: dA/dR =     0.6500
  t= 5: dA/dR =     0.5845
  t= 6: dA/dR =     0.5264
  t= 7: dA/dR =     0.4747
  t= 8: dA/dR =     0.4285
  t= 9: dA/dR =     0.3871


## 3. Verify with finite-difference TM propagation

In [4]:
dx = 0.0001
base_Rfree = agent.Rfree[0].copy()

# Build a perturbed agent with Rfree shifted by dx in both Markov states.
# Re-solve and rebuild TM so the perturbed transition matrix reflects the
# new Rfree's effect on savings policy and thus on the distribution dynamics.
agent_pert = deepcopy(agent)
agent_pert.Rfree = [base_Rfree + dx]
agent_pert.neutral_measure = True
agent_pert.construct("IncShkDstn", "TranShkDstn", "PermShkDstn")
agent_pert.define_distribution_grid(dist_mGrid=agent.dist_mGrid)
agent_pert.calc_transition_matrix()

D_ss = agent.vec_erg_dstn.flatten()
M = len(agent.dist_mGrid)
J = 2

c_ss_flat = np.concatenate(agent.cPol_Grid)
a_ss_flat = np.concatenate(agent.aPol_Grid)

# FD IRF: apply the perturbed TM at t=0, then revert to the SS TM.
# NOTE: this loop transitions the distribution BEFORE computing aggregates,
# i.e. A_fd[t] = a' @ (TM @ dstn). If the Jacobian uses
# compute-then-transition ordering, this introduces an off-by-one shift.
dstn = D_ss.copy()
A_fd = np.zeros(T)
for t in range(T):
    tm = agent_pert.tran_matrix if t == 0 else agent.tran_matrix
    dstn = tm @ dstn
    A_fd[t] = np.dot(a_ss_flat, dstn)

dA_fd = (A_fd - A_ss) / dx

print("Finite-difference vs Jacobian (first column):")
print(f"{'t':>3s} {'AJAC[:,0]':>12s} {'FD dA/dR':>12s} {'Diff':>12s}")
for t in range(min(15, T)):
    print(f"{t:3d} {AJAC[t, 0]:12.4f} {dA_fd[t]:12.4f} {AJAC[t, 0] - dA_fd[t]:12.6f}")

# Restore original Rfree
agent.Rfree = [base_Rfree]

Finite-difference vs Jacobian (first column):
  t    AJAC[:,0]     FD dA/dR         Diff
  0       1.0131       0.7251     0.287984
  1       0.9033       0.6495     0.253735
  2       0.8077       0.5829     0.224749
  3       0.7239       0.5240     0.199826
  4       0.6500       0.4718     0.178187
  5       0.5845       0.4253     0.159273
  6       0.5264       0.3838     0.142656
  7       0.4747       0.3467     0.127998
  8       0.4285       0.3134     0.115023
  9       0.3871       0.2836     0.103504
 10       0.3501       0.2568     0.093251
 11       0.3168       0.2327     0.084105
 12       0.2870       0.2110     0.075931
 13       0.2601       0.1915     0.068612
 14       0.2358       0.1738     0.062049


### Known issue: ~28% Jacobian vs finite-difference disagreement

The Jacobian column and the FD derivative disagree by roughly 28% at every
horizon — the ratio `AJAC[t,0] / dA_fd[t]` is nearly constant (~1.40).
A *constant multiplicative* discrepancy rules out simple numerical noise and
points to a systematic timing or normalisation mismatch.

**Likely causes (in order of probability):**

1. **Off-by-one / transition-then-compute vs compute-then-transition.**
   The FD loop above transitions the distribution *before* computing
   aggregates: `dstn = TM @ dstn; A = a' @ dstn`. If `calc_jacobian`
   uses the opposite convention (compute aggregates from the *current*
   distribution, *then* transition), the FD IRF is effectively shifted
   forward by one period relative to the Jacobian.  
   This is the same class of bug identified as **Fix #6** in the
   `Transition_Matrix_Example` notebook.

2. **Perturbed-agent setup.** The FD agent perturbs `Rfree` and rebuilds
   the transition matrix, but uses the *steady-state* policy grids
   (`a_ss_flat`, `c_ss_flat`) to compute aggregates. A fully consistent
   FD check would also use the *perturbed* policy grids — the mismatch
   means the FD derivative captures only the "distribution channel" of the
   Rfree shock and misses the "policy channel."

3. **Incorrect order of operations in the FD loop.** Related to (1), the
   loop applies the perturbed TM at `t == 0` and then the SS TM for
   `t >= 1`, but the aggregate is always computed *after* the transition.
   This means `A_fd[0]` already reflects one full transition step, which
   may not align with the Jacobian's definition of the period-0 response.

Resolving this is tracked as a **Tier 1B** investigation item.

## 4. IRF plot description

In [5]:
# Characterize the IRF shape
irf_A = AJAC[:, 0]
irf_C = CJAC[:, 0]

print(
    f"Asset IRF: peak at t={np.argmax(np.abs(irf_A))}, peak value = {irf_A[np.argmax(np.abs(irf_A))]:.4f}"
)
print(
    f"Consumption IRF: peak at t={np.argmax(np.abs(irf_C))}, peak value = {irf_C[np.argmax(np.abs(irf_C))]:.4f}"
)
print(f"Asset IRF sign at t=0: {'positive' if irf_A[0] > 0 else 'negative'}")
print(f"Consumption IRF sign at t=0: {'positive' if irf_C[0] > 0 else 'negative'}")
print(
    f"Half-life of |asset IRF|: ~{np.argmax(np.abs(irf_A) < np.abs(irf_A).max() / 2)} periods"
)

Asset IRF: peak at t=0, peak value = 1.0131
Consumption IRF: peak at t=0, peak value = 0.1385
Asset IRF sign at t=0: positive
Consumption IRF sign at t=0: positive
Half-life of |asset IRF|: ~7 periods


## 5. Summary

The `calc_jacobian` method on `MarkovConsumerType` implements the
Fake News Algorithm for Markov models:

- **Speed**: Computing a 50×50 Jacobian takes only seconds.
- **Block structure**: The (M×J) × (M×J) transition matrices correctly
  handle cross-state transitions in the Markov model.
- **Open issue**: The Jacobian and finite-difference IRFs disagree by
  ~28% at every horizon. The discrepancy is multiplicatively constant,
  suggesting a systematic timing or normalisation mismatch (see the
  "Known issue" cell above). This must be resolved before the Jacobian
  can be considered fully validated.

This enables sequence-space analysis of heterogeneous-agent models with
discrete Markov states — a key building block for HANK models with
state-dependent dynamics.
